In [ ]:
# Cell 1: Install all required packages
!pip install streamlit langchain faiss-cpu pypdf2 pdf2image pytesseract pillow python-docx pymupdf groq psutil pdfplumber unstructured sentence-transformers langchain-community langchain-huggingface

# Install system dependencies
!apt-get install poppler-utils -y
!apt-get install tesseract-ocr -y
!apt-get install libtesseract-dev -y

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 27.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.7/64.7 kB 5.7 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.32.5 which is incompatible.
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
poppler-utils is already the newest version (22.02.0-2ubuntu0.10).
0 upgraded, 0 newly installed, 0 to remove and 38 not upgraded.
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
tesseract-ocr is already the newest version (4.1.1-2.1build1).
0 upgraded, 0 newly i

In [ ]:
# Cell 2: Upload your knowledge_base files
from google.colab import files
import os
import zipfile

# # Create knowledge_base directory
# os.makedirs('knowledge_base', exist_ok=True)

# # Upload your PDF files (you can select multiple files)
# uploaded = files.upload()

# # If you have a zip file with all your documents
uploaded = files.upload()  # Upload your zip file
with zipfile.ZipFile('knowledge_base.zip', 'r') as zip_ref:
    zip_ref.extractall('')

KeyboardInterrupt: 

In [ ]:
# Cell 3: Fixed document processing for Colab
import os
import pymupdf  # This is fitz - using correct import
from pdf2image import convert_from_path
import pytesseract
from PIL import Image, ImageEnhance
from langchain_community.document_loaders import PDFPlumberLoader, Docx2txtLoader, TextLoader
from langchain.text_splitter import CharacterTextSplitter
import gc

# Colab-specific paths
POPPLER_PATH = "/usr/bin"
TESSERACT_CMD = "/usr/bin/tesseract"

# Configure pytesseract
pytesseract.pytesseract.tesseract_cmd = TESSERACT_CMD

def is_scanned_pdf(file_path):
    """Check if PDF is scanned (image-based)"""
    try:
        with pymupdf.open(file_path) as doc:  # Using pymupdf instead of fitz
            if len(doc) == 0:
                return True

            # Check first few pages for text
            pages_to_check = min(3, len(doc))
            text_pages = 0

            for i in range(pages_to_check):
                text = doc[i].get_text().strip()
                if len(text) > 100:  # Reasonable text threshold
                    text_pages += 1

            return text_pages == 0  # No text = scanned PDF

    except Exception as e:
        print(f"PDF analysis error: {str(e)}")
        return True

def colab_ocr_pdf(file_path):
    """OCR processing for scanned PDFs in Colab"""
    try:
        print(f"🔍 OCR Processing: {os.path.basename(file_path)}")

        # Convert PDF to images
        images = convert_from_path(
            file_path,
            dpi=300,
            poppler_path=POPPLER_PATH,
            grayscale=True,
        )

        full_text = []
        for i, image in enumerate(images, 1):
            try:
                # OCR the image
                text = pytesseract.image_to_string(
                    image,
                    lang="eng",
                    config="--psm 6"
                )

                if text.strip():
                    full_text.append(f"Page {i}:\n{text}")
                    print(f"✅ OCR Page {i}")

                # Clean up
                del image
                gc.collect()

            except Exception as e:
                print(f"❌ Page {i} OCR failed: {str(e)}")
                continue

        if full_text:
            # Save OCR text
            ocr_text = "\n\n".join(full_text)
            print(f"✅ OCR extracted {len(full_text)} pages")
            return ocr_text
        else:
            print("❌ No text extracted via OCR")
            return None

    except Exception as e:
        print(f"❌ OCR processing failed: {str(e)}")
        return None

def colab_load_document(file_path):
    """Complete document loader for Colab"""
    if not os.path.exists(file_path):
        print(f"❌ File not found: {file_path}")
        return None

    file_ext = os.path.splitext(file_path)[1].lower()
    print(f"📄 Processing: {os.path.basename(file_path)}")

    try:
        if file_ext == ".pdf":
            # Check if scanned PDF
            if is_scanned_pdf(file_path):
                print("🖼️  Scanned PDF detected - using OCR")
                ocr_text = colab_ocr_pdf(file_path)

                if ocr_text:
                    # Create document from OCR text
                    from langchain.schema import Document
                    docs = [Document(page_content=ocr_text)]

                    splitter = CharacterTextSplitter(
                        chunk_size=800,
                        chunk_overlap=100,
                        separator="\n\n"
                    )
                    chunks = splitter.split_documents(docs)
                    print(f"✅ OCR split into {len(chunks)} chunks")
                    return chunks
                else:
                    return None
            else:
                # Text-based PDF
                print("📝 Text-based PDF - direct extraction")
                loader = PDFPlumberLoader(file_path)
                docs = loader.load()

                if docs:
                    splitter = CharacterTextSplitter(
                        chunk_size=1000,
                        chunk_overlap=150,
                        separator="\n\n"
                    )
                    chunks = splitter.split_documents(docs)
                    print(f"✅ PDF split into {len(chunks)} chunks")
                    return chunks
                return docs

        elif file_ext == ".docx":
            print("📝 Processing DOCX file")
            loader = Docx2txtLoader(file_path)
            docs = loader.load()

            if docs:
                splitter = CharacterTextSplitter(
                    chunk_size=1000,
                    chunk_overlap=150
                )
                chunks = splitter.split_documents(docs)
                print(f"✅ DOCX split into {len(chunks)} chunks")
                return chunks
            return docs

        elif file_ext == ".txt":
            print("📄 Processing TXT file")
            loader = TextLoader(file_path, encoding="utf-8")
            docs = loader.load()

            if docs:
                splitter = CharacterTextSplitter(
                    chunk_size=1000,
                    chunk_overlap=150
                )
                chunks = splitter.split_documents(docs)
                print(f"✅ TXT split into {len(chunks)} chunks")
                return chunks
            return docs

        else:
            print(f"❌ Unsupported file type: {file_ext}")
            return None

    except Exception as e:
        print(f"❌ Error loading {file_path}: {str(e)}")
        return None

In [ ]:
# Cell 4: Train on all documents
def colab_train_all_documents():
    """Train on all documents in Colab"""
    knowledge_base_dir = "knowledge_base"

    if not os.path.exists(knowledge_base_dir):
        print("❌ Knowledge base directory not found!")
        return []

    # Get all files
    all_files = []
    for root, dirs, files in os.walk(knowledge_base_dir):
        for file in files:
            if file.lower().endswith(('.pdf', '.docx', '.txt')):
                full_path = os.path.join(root, file)
                all_files.append(full_path)

    print(f"📚 Found {len(all_files)} files to process")
    print("Files:", [os.path.basename(f) for f in all_files])

    all_documents = []

    for i, file_path in enumerate(all_files, 1):
        print(f"\n{'='*60}")
        print(f"🔄 Processing {i}/{len(all_files)}: {os.path.basename(file_path)}")

        docs = colab_load_document(file_path)
        if docs:
            all_documents.extend(docs)
            print(f"✅ Added {len(docs)} chunks (Total: {len(all_documents)})")
        else:
            print("⚠️ No documents extracted")

    print(f"\n🎯 TRAINING COMPLETED!")
    print(f"📊 Total documents: {len(all_documents)}")

    # Calculate total text size
    total_size = sum(len(d.page_content) for d in all_documents) / 1024 / 1024
    print(f"💾 Total text size: {total_size:.2f} MB")

    return all_documents

# Start training
print("🚀 Starting training in Google Colab...")
documents = colab_train_all_documents()

🚀 Starting training in Google Colab...
📚 Found 20 files to process
Files: ['2022081640-1.pdf', 'universal_declaration_of_human_rights.pdf', '2022080893-2.pdf', '2022081025.pdf', 'ImpCaseLawsRuchiMajoo.pdf', 'Labour-Law-Reforms-Book-NLU-Delhi-2021.pdf', '2022080815.pdf', 'Labour Act (1).pdf', '2022080534.pdf', 'labour_code_eng.pdf', '2022080535.pdf', '2022080562.pdf', '2022080589-1.pdf', 'Handbook-on-New-Labour-Codes.pdf', 'NRI_Imp_CaseLaws.pdf', 'SBAA7017.pdf', 'Labour_Laws&_Practice.pdf', 'labour-employment-law-india.pdf', '2022080584-2.pdf', '2022081053-2.pdf']

🔄 Processing 1/20: 2022081640-1.pdf
📄 Processing: 2022081640-1.pdf
📝 Text-based PDF - direct extraction
✅ PDF split into 41 chunks
✅ Added 41 chunks (Total: 41)

🔄 Processing 2/20: universal_declaration_of_human_rights.pdf
📄 Processing: universal_declaration_of_human_rights.pdf
📝 Text-based PDF - direct extraction
✅ PDF split into 8 chunks
✅ Added 8 chunks (Total: 49)

🔄 Processing 3/20: 2022080893-2.pdf
📄 Processing: 2022080

✅ OCR Page 78
✅ OCR extracted 78 pages
✅ OCR split into 271 chunks
✅ Added 271 chunks (Total: 320)

🔄 Processing 4/20: 2022081025.pdf
📄 Processing: 2022081025.pdf
📝 Text-based PDF - direct extraction
✅ PDF split into 39 chunks
✅ Added 39 chunks (Total: 359)

🔄 Processing 5/20: ImpCaseLawsRuchiMajoo.pdf
📄 Processing: ImpCaseLawsRuchiMajoo.pdf
📝 Text-based PDF - direct extraction
✅ PDF split into 1 chunks
✅ Added 1 chunks (Total: 360)

🔄 Processing 6/20: Labour-Law-Reforms-Book-NLU-Delhi-2021.pdf
📄 Processing: Labour-Law-Reforms-Book-NLU-Delhi-2021.pdf
📝 Text-based PDF - direct extraction
✅ PDF split into 473 chunks
✅ Added 473 chunks (Total: 833)

🔄 Processing 7/20: 2022080815.pdf
📄 Processing: 2022080815.pdf
🖼️  Scanned PDF detected - using OCR
🔍 OCR Processing: 2022080815.pdf
✅ OCR Page 1
✅ OCR Page 2
✅ OCR Page 3
✅ OCR Page 4
✅ OCR Page 5
✅ OCR Page 6
✅ OCR Page 7
✅ OCR Page 8
✅ OCR Page 9
✅ OCR Page 10
✅ OCR Page 11
✅ OCR Page 12
✅ OCR Page 13


✅ OCR Page 14
✅ OCR extracted 14 pages
✅ OCR split into 33 chunks
✅ Added 33 chunks (Total: 866)

🔄 Processing 8/20: Labour Act (1).pdf
📄 Processing: Labour Act (1).pdf
📝 Text-based PDF - direct extraction
✅ PDF split into 197 chunks
✅ Added 197 chunks (Total: 1063)

🔄 Processing 9/20: 2022080534.pdf
📄 Processing: 2022080534.pdf
🖼️  Scanned PDF detected - using OCR
🔍 OCR Processing: 2022080534.pdf
✅ OCR Page 1
✅ OCR Page 2
✅ OCR Page 3
✅ OCR Page 4
✅ OCR Page 5
✅ OCR Page 6
✅ OCR Page 7
✅ OCR Page 8
✅ OCR Page 9
✅ OCR Page 10
✅ OCR Page 11
✅ OCR Page 12
✅ OCR Page 13
✅ OCR Page 14
✅ OCR Page 15
✅ OCR Page 16
✅ OCR Page 17
✅ OCR Page 18
✅ OCR Page 19
✅ OCR Page 20
✅ OCR Page 21
✅ OCR Page 22
✅ OCR Page 23
✅ OCR Page 24
✅ OCR Page 25
✅ OCR Page 26
✅ OCR Page 27
✅ OCR Page 28
✅ OCR Page 29
✅ OCR Page 30
✅ OCR Page 31
✅ OCR Page 32
✅ OCR Page 33
✅ OCR Page 34
✅ OCR Page 35
✅ OCR Page 36
✅ OCR Page 37
✅ OCR Page 38
✅ OCR Page 39
✅ OCR Page 40
✅ OCR Page 41
✅ OCR Page 42
✅ OCR Page 43
✅ OCR 

✅ OCR Page 167
✅ OCR extracted 167 pages
✅ OCR split into 351 chunks
✅ Added 351 chunks (Total: 1414)

🔄 Processing 10/20: labour_code_eng.pdf
📄 Processing: labour_code_eng.pdf
📝 Text-based PDF - direct extraction
✅ PDF split into 36 chunks
✅ Added 36 chunks (Total: 1450)

🔄 Processing 11/20: 2022080535.pdf
📄 Processing: 2022080535.pdf
🖼️  Scanned PDF detected - using OCR
🔍 OCR Processing: 2022080535.pdf
✅ OCR Page 1
✅ OCR Page 2
✅ OCR Page 3
✅ OCR Page 4
✅ OCR Page 5
✅ OCR Page 6
✅ OCR Page 7
✅ OCR Page 8
✅ OCR Page 9
✅ OCR Page 10
✅ OCR Page 11
✅ OCR Page 12
✅ OCR Page 13
✅ OCR Page 14
✅ OCR Page 15
✅ OCR Page 16
✅ OCR Page 17
✅ OCR Page 18
✅ OCR Page 19


✅ OCR Page 20
✅ OCR extracted 20 pages
✅ OCR split into 48 chunks
✅ Added 48 chunks (Total: 1498)

🔄 Processing 12/20: 2022080562.pdf
📄 Processing: 2022080562.pdf
🖼️  Scanned PDF detected - using OCR
🔍 OCR Processing: 2022080562.pdf
✅ OCR Page 1
✅ OCR Page 2
✅ OCR Page 3
✅ OCR Page 4
✅ OCR Page 5
✅ OCR Page 6
✅ OCR Page 7
✅ OCR Page 8
✅ OCR Page 9
✅ OCR Page 10
✅ OCR Page 11
✅ OCR Page 12
✅ OCR Page 13
✅ OCR Page 14
✅ OCR Page 15
✅ OCR Page 16
✅ OCR Page 17
✅ OCR Page 18
✅ OCR Page 19
✅ OCR Page 20
✅ OCR Page 21
✅ OCR Page 22
✅ OCR Page 23
✅ OCR Page 24
✅ OCR Page 25
✅ OCR Page 26
✅ OCR Page 27
✅ OCR Page 28
✅ OCR Page 29
✅ OCR Page 30
✅ OCR Page 31
✅ OCR Page 32
✅ OCR Page 33
✅ OCR Page 34
✅ OCR Page 35
✅ OCR Page 36
✅ OCR Page 37
✅ OCR Page 38
✅ OCR Page 39
✅ OCR Page 40
✅ OCR Page 41
✅ OCR Page 42
✅ OCR Page 43
✅ OCR Page 44
✅ OCR Page 45
✅ OCR Page 46
✅ OCR Page 47
✅ OCR Page 48
✅ OCR Page 49
✅ OCR Page 50
✅ OCR Page 51
✅ OCR Page 52
✅ OCR Page 53
✅ OCR Page 54
✅ OCR Page 55
✅ OCR 

✅ OCR Page 131
✅ OCR extracted 131 pages
✅ OCR split into 238 chunks
✅ Added 238 chunks (Total: 1736)

🔄 Processing 13/20: 2022080589-1.pdf
📄 Processing: 2022080589-1.pdf
🖼️  Scanned PDF detected - using OCR
🔍 OCR Processing: 2022080589-1.pdf
✅ OCR Page 1
✅ OCR Page 2
✅ OCR Page 3
✅ OCR Page 4
✅ OCR Page 5
✅ OCR Page 6
✅ OCR Page 7
✅ OCR Page 8
✅ OCR Page 9
✅ OCR Page 10
✅ OCR Page 11
✅ OCR Page 12
✅ OCR Page 13
✅ OCR Page 14
✅ OCR Page 15
✅ OCR Page 16
✅ OCR Page 17
✅ OCR Page 18
✅ OCR Page 19
✅ OCR Page 20
✅ OCR Page 21
✅ OCR Page 22
✅ OCR Page 23
✅ OCR Page 24
✅ OCR Page 25
✅ OCR Page 26
✅ OCR Page 27


✅ OCR Page 28
✅ OCR extracted 28 pages
✅ OCR split into 63 chunks
✅ Added 63 chunks (Total: 1799)

🔄 Processing 14/20: Handbook-on-New-Labour-Codes.pdf
📄 Processing: Handbook-on-New-Labour-Codes.pdf
📝 Text-based PDF - direct extraction
✅ PDF split into 52 chunks
✅ Added 52 chunks (Total: 1851)

🔄 Processing 15/20: NRI_Imp_CaseLaws.pdf
📄 Processing: NRI_Imp_CaseLaws.pdf
📝 Text-based PDF - direct extraction
✅ PDF split into 20 chunks
✅ Added 20 chunks (Total: 1871)

🔄 Processing 16/20: SBAA7017.pdf
📄 Processing: SBAA7017.pdf
🖼️  Scanned PDF detected - using OCR
🔍 OCR Processing: SBAA7017.pdf
✅ OCR Page 1
✅ OCR Page 2
✅ OCR Page 3
✅ OCR Page 4


✅ OCR Page 5
✅ OCR extracted 5 pages
✅ OCR split into 8 chunks
✅ Added 8 chunks (Total: 1879)

🔄 Processing 17/20: Labour_Laws&_Practice.pdf
📄 Processing: Labour_Laws&_Practice.pdf
📝 Text-based PDF - direct extraction


✅ PDF split into 568 chunks
✅ Added 568 chunks (Total: 2447)

🔄 Processing 18/20: labour-employment-law-india.pdf
📄 Processing: labour-employment-law-india.pdf
📝 Text-based PDF - direct extraction
✅ PDF split into 24 chunks
✅ Added 24 chunks (Total: 2471)

🔄 Processing 19/20: 2022080584-2.pdf
📄 Processing: 2022080584-2.pdf
🖼️  Scanned PDF detected - using OCR
🔍 OCR Processing: 2022080584-2.pdf
✅ OCR Page 1
✅ OCR Page 2
✅ OCR Page 3
✅ OCR Page 4
✅ OCR Page 5
✅ OCR Page 6
✅ OCR Page 7
✅ OCR Page 8
✅ OCR Page 9
✅ OCR Page 10
✅ OCR Page 11
✅ OCR Page 12
✅ OCR Page 13
✅ OCR Page 14
✅ OCR Page 15
✅ OCR Page 16
✅ OCR Page 17
✅ OCR Page 18
✅ OCR Page 19
✅ OCR Page 20
✅ OCR Page 21
✅ OCR Page 22


✅ OCR Page 23
✅ OCR extracted 23 pages
✅ OCR split into 80 chunks
✅ Added 80 chunks (Total: 2551)

🔄 Processing 20/20: 2022081053-2.pdf
📄 Processing: 2022081053-2.pdf
📝 Text-based PDF - direct extraction
✅ PDF split into 42 chunks
✅ Added 42 chunks (Total: 2593)

🎯 TRAINING COMPLETED!
📊 Total documents: 2593
💾 Total text size: 4.24 MB


In [ ]:
# Cell 5: Create and save FAISS vector store
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings

def create_vector_store(documents):
    """Create FAISS vector store"""
    if not documents:
        print("❌ No documents to create vector store")
        return None

    print("🔄 Creating embeddings...")

    # Use efficient model for Colab
    embeddings = HuggingFaceEmbeddings(
        model_name="sentence-transformers/all-MiniLM-L6-v2",
        model_kwargs={'device': 'cpu'}
    )

    print("🔄 Creating FAISS vector store...")
    vector_store = FAISS.from_documents(documents, embeddings)

    print("💾 Saving vector store...")
    vector_store.save_local("faiss_index")

    print("✅ Vector store created and saved!")
    return vector_store

# Create vector store
if documents:
    vector_store = create_vector_store(documents)
else:
    print("❌ No documents to process")

🔄 Creating embeddings...
🔄 Creating FAISS vector store...


KeyboardInterrupt: 

In [ ]:
# Cell 6: Download the trained model
from google.colab import files

# Create zip of the FAISS index
!zip -r faiss_index_colab.zip faiss_index/

# Download
print("📥 Downloading FAISS index...")
files.download('faiss_index_colab.zip')

print("✅ Download complete! Use this in your local Streamlit app.")

  adding: faiss_index/ (stored 0%)
  adding: faiss_index/index.pkl (deflated 68%)
  adding: faiss_index/index.faiss (deflated 7%)
📥 Downloading FAISS index...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✅ Download complete! Use this in your local Streamlit app.
